**#04. Text Parsing, Filtering, and Quantitative Representation**

- **Instructor:** Jaeung Sim (University of Connecticut)
- **Course:** OPIM 5671 Data Mining and Time Series Forecasting
- **Last updated:** October 27, 2026

This hands-on exercise applies the week 8 lecture material. Complete it
before the full week 9 slide deck and combined Notebook 06.
We retain the original YouTube comments dataset and the progression from text
processing to TF–IDF and singular value decomposition (SVD).

**Learning goals:** Explain how cleaning changes text, read a document–term
matrix, and interpret a few SVD components without treating them as known topics.

| Activity | Minutes |
|---|---:|
| Colab setup and data | 5 |
| Parsing and filtering | 7 |
| TF–IDF interpretation | 8 |
| SVD interpretation | 8 |
| Bridge to week 9 | 2 |
| Total | 30 |

Use Google Colab's default CPU runtime and run cells in order. The working sample
is capped at 2,000 comments, TF–IDF at 2,000 terms, and SVD at 10 components.
Initial downloads depend on network availability; there is no model search or GPU setup.


## 1. Colab setup and data
Install KaggleHub and NLTK if needed. NumPy, pandas, matplotlib and scikit-learn
use Colab's existing versions. We download only the stopword list and WordNet.
`preserve_line=True` tokenizes without sentence segmentation, so no Punkt
sentence-tokenizer download is needed.


In [ ]:
%pip -q install kagglehub nltk


In [ ]:
import re, time, importlib.metadata
from pathlib import Path
from collections import Counter
from itertools import chain
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from IPython.display import display

SEED = 25
MAX_COMMENTS = 2000
MAX_TERMS = 2000
MAX_COMPONENTS = 10
pd.set_option('display.max_colwidth', 110)
for package in ['pandas','scikit-learn','nltk']:
    print(package, importlib.metadata.version(package))


In [ ]:
for resource in ['stopwords','wordnet']:
    nltk.download(resource, quiet=True, raise_on_error=True)


### Inspect the YouTube comments
Source: [YouTube Comments Dataset](https://www.kaggle.com/datasets/atifaliak/youtube-comments-dataset).
Treat `Comment` as one document per row. Even a dataset described as cleaned
still needs missing-value checks and inspection for the current task.
If the automatic download is unavailable, upload the dataset CSV in Colab's Files
pane and set `LOCAL_CSV` to its `/content/...csv` path below.


In [ ]:
LOCAL_CSV = None
if LOCAL_CSV:
    csv_path = Path(LOCAL_CSV)
else:
    import kagglehub
    data_path = Path(kagglehub.dataset_download('atifaliak/youtube-comments-dataset'))
    candidates = sorted(data_path.rglob('*.csv'))
    if not candidates:
        raise FileNotFoundError('Upload the dataset CSV and set LOCAL_CSV.')
    # Select by schema rather than assuming the first CSV is the right one.
    matches = [p for p in candidates if 'Comment' in pd.read_csv(p,nrows=0).columns]
    if len(matches) != 1:
        raise ValueError('Set LOCAL_CSV to the CSV containing Comment.')
    csv_path = matches[0]
raw = pd.read_csv(csv_path)
assert 'Comment' in raw.columns, 'The CSV must contain a Comment column.'
print('Source rows:', len(raw), '| Missing comments:', raw['Comment'].isna().sum())
df = raw.dropna(subset=['Comment']).copy()
df['Comment'] = df['Comment'].astype(str).str.strip()
df = df.loc[df['Comment'].ne('')].drop_duplicates(subset=['Comment'])
df = df.sample(n=min(MAX_COMMENTS,len(df)),random_state=SEED).reset_index(drop=True)
if len(df) < 3:
    raise ValueError('At least three nonempty comments are needed.')
print('Working sample:', len(df), 'comments')
display(df[['Comment']].head())


## 2. Parsing and filtering
Tokenization separates text into tokens. For this English vocabulary exercise,
we lowercase, replace nonletters with spaces, remove English stopwords and
apply WordNet's default **noun** lemmatization. Original text remains in `Comment`.

Lemmatization depends on part of speech: the default can turn “cars” into “car”
but may leave “running” unchanged. Mapping “running” to “run” requires specifying
its verb use. We do not add a tagging pipeline in this short exercise.


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def tokenize_text(text):
    return word_tokenize(str(text), preserve_line=True)

def preprocess_text(text):
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower())
    tokens = tokenize_text(text)
    return ' '.join(lemmatizer.lemmatize(w) for w in tokens
                    if w not in stop_words and len(w)>1)

df['tokenized'] = df['Comment'].map(tokenize_text)
df['processed'] = df['Comment'].map(preprocess_text)
display(df[['Comment','tokenized','processed']].head(3))
print('Default noun lemma:', lemmatizer.lemmatize('cars'))
print('Verb lemma when specified:', lemmatizer.lemmatize('running',pos='v'))


### Cleaning depends on the task
This simple representation drops emoji, punctuation, numbers and non-English
letters. It can also remove negation. Those are deliberate simplifications here,
but can discard useful sentiment cues. Keep original text when moving to sentiment
analysis in Notebook 06, where “not good” should remain available to the scorer.


In [ ]:
examples = ['This video is good!', 'This video is not good!']
display(pd.DataFrame({'Original':examples,
    'After filtering':[preprocess_text(t) for t in examples]}))


### Frequent terms before and after filtering
Compare the 20 most frequent tokens instead of a crowded 100-term plot. The first
view includes case differences and punctuation. The second uses filtered words.
This is a frequency comparison; a bar chart alone does not establish Zipf's law.


In [ ]:
before = Counter(chain.from_iterable(df['tokenized']))
after = Counter(chain.from_iterable(df['processed'].str.split()))
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax, counts, title in zip(axes,[before,after],['Before filtering','After filtering']):
    top = counts.most_common(20)
    if top:
        words, frequencies = zip(*reversed(top))
        ax.barh(words,frequencies)
    ax.set_title(title)
    ax.set_xlabel('Token count')
plt.tight_layout(); plt.show()
print('Unique raw tokens:',len(before),'| Unique filtered tokens:',len(after))


**Quick check:** Name one kind of information removed by filtering. Could it be
useful for another text task?


## 3. TF–IDF: reading the representation (8 minutes)
TF–IDF gives more weight to terms that occur in a document and are less widespread
across the corpus. A high weight indicates a term distinctive within this
representation; it is not a sentiment score or a universal importance measure.

Here each **row is a comment** and each **column is a vocabulary term**. Most
entries are zero. Keep the full matrix sparse. We show only a small dense slice.
`min_df=2` removes terms occurring in fewer than two comments; `max_df=0.95`
removes terms in more than 95% of comments; `max_features=2000` bounds the vocabulary.
The default vectorizer also excludes single-character tokens and normalizes rows.


In [ ]:
# Drop comments that contain no words after filtering, preserving row alignment.
empty_count = int(df['processed'].str.strip().eq('').sum())
df = df.loc[df['processed'].str.strip().ne('')].reset_index(drop=True)
print('Comments removed after filtering:',empty_count)
vectorizer = TfidfVectorizer(min_df=2,max_df=0.95,max_features=MAX_TERMS)
X_tfidf = vectorizer.fit_transform(df['processed'])
terms = vectorizer.get_feature_names_out()
assert X_tfidf.shape[0] == len(df)
print('Document–term matrix:',X_tfidf.shape)
print('Retained vocabulary:',len(terms))
print(f'Zero entries: {100*(1-X_tfidf.nnz/np.prod(X_tfidf.shape)):.1f}%')
print('Nonempty comments with no retained terms:',int((X_tfidf.getnnz(axis=1)==0).sum()))


### Inspect one comment's high-weight terms
The next cell chooses a comment with at least four retained terms when available.
Read its original text alongside its eight largest TF–IDF weights.
Why might these words distinguish it from other comments?


In [ ]:
term_counts = X_tfidf.getnnz(axis=1)
candidates = np.flatnonzero(term_counts >= 4)
example_row = int(candidates[0] if len(candidates) else np.argmax(term_counts))
print(df.loc[example_row,'Comment'])
row = X_tfidf.getrow(example_row)
order = np.argsort(row.data)[::-1][:8]
example_terms = terms[row.indices[order]]
display(pd.DataFrame({'Term':example_terms,'TF_IDF':row.data[order]}).round(3))
# Dense conversion is limited to this 5-by-at-most-8 illustration.
small = X_tfidf[:5, row.indices[order]].toarray()
display(pd.DataFrame(small,index=[f'Comment {i}' for i in range(5)],
                     columns=example_terms).round(3))


**Quick check:** What does a zero mean? Why can the vocabulary size differ from
the number of unique filtered tokens? Refer to the vectorizer's filtering rules
and vocabulary cap rather than assuming a fixed count from a previous run.


## 4. SVD: compact features and interpretation (8 minutes)
Truncated SVD approximates the document–term matrix with a small number of
components. Each component combines terms; it does **not** simply select a few
original words. Each comment receives a score on each component.
SVD operates directly on the sparse TF–IDF matrix and produces a small dense
matrix of comment scores. We cap it at ten components for this exercise.


In [ ]:
n_components = min(MAX_COMPONENTS,X_tfidf.shape[0]-1,X_tfidf.shape[1]-1)
if n_components < 2:
    raise ValueError('Use more comments so that at least two components are available.')
started = time.perf_counter()
svd = TruncatedSVD(n_components=n_components,random_state=SEED)
X_svd = svd.fit_transform(X_tfidf)
component_names = [f'Component_{i+1}' for i in range(n_components)]
df_svd = pd.DataFrame(X_svd,index=df.index,columns=component_names)
print(f'{X_tfidf.shape[1]} term columns become {n_components} component columns.')
print(f'SVD fit: {time.perf_counter()-started:.2f} seconds')
display(df_svd.head().round(3))
cumulative = np.cumsum(svd.explained_variance_ratio_)
print(f'Cumulative explained variance: {cumulative[-1]:.1%}')
plt.plot(np.arange(1,n_components+1),cumulative,marker='o')
plt.xticks(np.arange(1,n_components+1))
plt.xlabel('Components retained'); plt.ylabel('Cumulative explained variance')
plt.title('Compression retains only part of the original variation')
plt.show()


The reported explained variance describes variation in this TF–IDF representation,
not prediction accuracy or a percentage of meaning understood. Ten components
may retain only a modest share of the variation; we use that cap to demonstrate
compression, not to claim an optimal choice.

### Inspect terms that define the components
Term loadings describe how words contribute to a component. Inspect both ends
of the first three components. Some components may have no negative loadings.
Component signs can flip without changing the underlying representation, so
“negative” here does **not** mean negative sentiment.


In [ ]:
loading_rows = []
for i, weights in enumerate(svd.components_[:3]):
    positive = np.flatnonzero(weights > 0)
    negative = np.flatnonzero(weights < 0)
    high = positive[np.argsort(weights[positive])[-5:][::-1]]
    low = negative[np.argsort(weights[negative])[:5]]
    loading_rows.append({'Component':component_names[i],
        'Largest positive loadings':', '.join(terms[high]),
        'Most negative loadings':', '.join(terms[low]) or '(none)'})
display(pd.DataFrame(loading_rows))


### Check the interpretation against actual comments
Inspect the two highest and two lowest scores on **Component 2**. Compare these
comments with its loading words above. A pattern might suggest a tentative label,
but it can also be difficult to interpret. SVD components are continuous features,
not guaranteed human-readable topics or exclusive document categories.


In [ ]:
component = 'Component_2'
scores = df_svd[component]
ids = list(dict.fromkeys(list(scores.nlargest(2).index)+list(scores.nsmallest(2).index)))
display(df.loc[ids,['Comment','processed']].join(df_svd[[component]]).round(3))


**Quick check:** Propose a tentative interpretation of Component 2, or explain why
you cannot name it confidently. Support your answer with words and example comments.


## 5. Bridge to week 9

| Representation | What its columns capture |
|---|---|
| TF–IDF | Weighted vocabulary terms |
| SVD scores | Combinations of terms that compress the TF–IDF matrix |
| Sentiment and emotion features in Notebook 06 | Polarity and lexical emotion associations |
| LDA topic weights in Notebook 06 | Each review's mixture of learned word patterns |

Next, the **whole week 9 slide deck** introduces predictive assessment and the
new feature types. Then combined Notebook 06 uses Disneyland reviews to test
whether these features improve rating predictions. LDA uses **word counts**,
while this SVD exercise uses TF–IDF. SVD scores are not required in Assignment #2.

Here we fit TF–IDF and SVD on the working corpus to explore its representation.
If using either in prediction, split first, fit both on training text, and use
their `.transform()` methods on validation/test text. Keep original text for
sentiment so that filtering does not erase cues such as negation.


## References
- [Original YouTube dataset](https://www.kaggle.com/datasets/atifaliak/youtube-comments-dataset)
- [NLTK word tokenization](https://www.nltk.org/api/nltk.tokenize.html)
- [WordNet lemmatization](https://www.nltk.org/api/nltk.stem.wordnet.html)
- [scikit-learn TF–IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [scikit-learn TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)
